# 02 · Baseline — Blob Detection + Hungarian Linking + Bölünme

**Sürüm 2** (ilk submit: edge_J 0.781 / **LB 0.749**). Bu turda test edilenler:
- ❌ **Sınır temizleme** — ZARARLI çıktı (edge_J 0.781→0.686; sınır node'ları gerçek hücre). KAPALI.
- ✅ **Bölünme detection** — dominant: edge_J +0.005, hiçbir metriği bozmuyor. AÇIK.

**Bu bir Code Competition.** Görünen `test/` bir *placeholder* (train'den kopyalanmış 4 film);
submit edince Kaggle **gerçek gizli test setini** bağlayıp bu defteri yeniden çalıştırır.
→ Dataset isimleri/sayısı **hardcode edilmez**, `test/` dinamik gezilir.

## Neden Ultrack değil de bu?
EDA (`01_eda`, `01b_eda_detailed`) şunları gösterdi:
- Hareket **< 8 µm** (95p 5.5 / 99p 8) → basit en-yakın-komşu linking çoğu kenarı doğru bağlar
- Kare-içi GT komşuları ~25 µm uzakta → **eşleşme belirsizliği yok**
- Zamansal **boşluk yok** → gap-closing gerekmez
- Çekirdekler arka plandan ~8× parlak → detection kolay

Yani %90'lık edge skorunun büyük kısmı **iyi detection + basit linking** ile erişilebilir.
`tracksdata`/`ultrack` kurulumu Kaggle'da numpy ABI'sini kırıyordu → **bağımlılık-hafif**,
kesin çalışan bir hat kuruyoruz. Ultrack sonraki upgrade.

## Parametreler (EDA'dan)
| Param | Değer | Kaynak |
|---|---|---|
| ölçek (Z,Y,X) µm/px | (1.625, 0.40625, 0.40625) | multiscales |
| linking yarıçapı | **8 µm** | D04 (99p) |
| metrik eşleşme | 7 µm | metrics.md |
| hedef yoğunluk | ~213 çekirdek/kare | D09 |

## 0 · Kurulum — ⚠️ internet KAPALI olmalı

**Doğrulandı** (temiz kernel + internet kapalı = submit ortamı):
`zarr`, `numcodecs`, `geff`, `tracksdata` → **YOK**. Mevcut: numpy 2.0.2, scipy 1.16.3,
skimage 0.25.2, pandas, networkx, dask, blosc2.

Submit için internet kapalı zorunlu → **zarr'ı bir Kaggle Dataset'i ile taşımalıyız** (tek seferlik):

1. **Yeni notebook** aç, **internet AÇIK**, tek hücre:
   ```
   !pip install --target=/kaggle/working/pylibs zarr
   ```
2. **Save & Run All** → sağdaki *Output* → **"New Dataset"** → ad: `cell-tracking-libs`
3. **Bu notebook'ta:** *Add Data* → `cell-tracking-libs` ekle
4. *Settings* → **Internet: Off** → *Save & Run All* → **Submit**

> Aynı Kaggle imajında kurulduğu için derlenmiş uzantılar (numcodecs) Python 3.12 + numpy 2.x
> ile uyumlu olur. Aşağıdaki hücre klasörü `/kaggle/input` altında **otomatik bulur**.

In [ ]:
# Bu yarismada SUBMIT icin internet KAPALI olmali; ama zarr base imajda YOK.
# Cozum: zarr wheel'leri bir Kaggle Dataset'inden offline kurulur.
# (Wheel dataset'i nasil hazirlanir -> asagidaki markdown'a bak.)
import sys, subprocess, glob, os

def _scan(base):
    # /kaggle/input altini gez; yarisma verisine (binlerce chunk) DALMA.
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs
                   if not d.endswith((".zarr", ".geff")) and d != "competitions"]
        if root.count(os.sep) > 9:
            dirs[:] = []; continue
        yield root, files

def ensure_zarr():
    try:
        import zarr; return zarr                      # zaten varsa
    except ImportError:
        pass
    # NOT: Kaggle mount duzeni /kaggle/input/datasets/<user>/<slug>/... olabiliyor
    # -> sabit glob yerine ARAMA yapiyoruz.
    # 1) sys.path: `pip install --target` ile hazirlanmis lib klasoru (pip'e gerek yok)
    for root, files in _scan("/kaggle/input"):
        if os.path.basename(root) == "zarr" and "__init__.py" in files:
            p = os.path.dirname(root)
            sys.path.insert(0, p)
            try:
                import zarr; print("zarr <- sys.path:", p); return zarr
            except ImportError:
                sys.path.pop(0)
    # 2) offline wheel klasoru (--no-index)
    for root, files in _scan("/kaggle/input"):
        if any(f.endswith(".whl") for f in files):
            subprocess.run([sys.executable,"-m","pip","install","-q","--no-index",
                            f"--find-links={root}","zarr"], capture_output=True)
            try:
                import zarr; print("zarr <- wheel:", root); return zarr
            except ImportError:
                pass
    # 3) ONLINE: sadece editorde gelistirirken (SUBMIT'te internet kapali -> calismaz)
    print("[uyari] offline kaynak yok -> internetten kurmayi deniyorum (SUBMIT'te CALISMAZ!)")
    subprocess.run([sys.executable,"-m","pip","install","-q","zarr"], check=False)
    import zarr; return zarr

zarr = ensure_zarr()
print("zarr:", zarr.__version__)

import time, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
warnings.filterwarnings("ignore")
print("hazir")

## 1 · Yapılandırma

In [ ]:
SCALE_ZYX = (1.625, 0.40625, 0.40625)   # um/piksel (Z,Y,X)
S = np.array(SCALE_ZYX, dtype=np.float32)

LINK_MAX_UM = 8.0      # linking arama yaricapi (EDA: 99p ~8um)
MATCH_UM    = 7.0      # metrik eslesme toleransi

SIGMA = (1, 2, 2)      # Gauss yumusatma (voxel; Z ince)
FOOT  = (3, 11, 11)    # yerel-maksimum footprint ~ cekirdek boyutu

# --- Sinir temizleme: TEST EDILDI, ZARARLI -> KAPALI (0).
#     Betul'un "sinir node'lari artefakt" varsayimi yanlis cikti: embriyo dokusu gorus
#     alaninin kenarina kadar uzaniyor, sinirdaki node'lar cogunlukla GERCEK hucre.
#     border=1 bile recall'i dusurdu (6bba_05b6850b 0.92->0.73, edge_J 0.781->0.686). ---
BORDER_MARGIN = 0      # KAPALI (kod parametreli kaldi ama devrede degil)

# --- YENI: bolunme (division) detection ---
DIV_ON      = True     # ikinci tur eslestirme ile bolunme kenarlari ekle
DIV_MAX_UM  = 6.0      # ebeveyn-kiz maks mesafe (EDA D07 medyani ~6um; linking gate'ten DAR)
DIV_SIB_UM  = 13.0     # iki kiz arasi maks mesafe (EDA D07 kiz-kiz 95p ~13um; FP filtresi)

MAX_TEST = None        # None = tum test datasetleri (gercek rerun icin SART)
OUT_CSV  = "/kaggle/working/submission.csv"

INPUT = Path("/kaggle/input")
def find_root():
    st=[(INPUT,0)]
    while st:
        b,d=st.pop()
        try:
            if (b/"train").is_dir() and (b/"test").is_dir(): return b
        except Exception: pass
        if d<4:
            for c in sorted(b.iterdir()):
                if c.is_dir() and not c.name.endswith((".zarr",".geff")): st.append((c,d+1))
ROOT=find_root(); TRAIN=ROOT/"train"; TEST=ROOT/"test"
train_names=sorted(p.stem for p in TRAIN.glob("*.zarr"))
test_names =sorted(p.stem for p in TEST.glob("*.zarr"))
print("ROOT:",ROOT)
print(f"train={len(train_names)} | test={len(test_names)}")
print("test:",test_names[:10], "..." if len(test_names)>10 else "")

# DEV mi GERCEK RERUN mu?
# Placeholder test/ = train'den kopyalanmis 4 film -> GT'leri train/'de var.
# Gercek rerun'da gizli test isimleri train'de YOKTUR -> tum dogrulama atlanmali.
DEV = len(test_names)>0 and (TRAIN/(test_names[0]+".geff")).exists()
print("\nMOD:", "DEV (placeholder test, GT var -> dogrulama yapilir)" if DEV
      else "GERCEK RERUN (gizli test, GT yok -> sadece submission uretilir)")

## 2 · Yardımcı okuyucular

In [ ]:
def open_image(zpath):
    n=zarr.open(str(zpath),mode="r"); a=dict(n.attrs)
    ms=a.get("multiscales")
    if ms is None and isinstance(a.get("ome"),dict): ms=a["ome"].get("multiscales")
    if ms:
        return n[ms[0]["datasets"][0]["path"]]
    keys=list(n.keys()) if hasattr(n,"keys") else []
    return n["0"] if "0" in keys else n

def load_geff(gp):
    g=zarr.open(str(gp),mode="r")
    nodes=g["nodes"]; ids=np.asarray(nodes["ids"]); props={}
    if "props" in nodes:
        for pn in list(nodes["props"].keys()):
            try: props[pn]=np.asarray(nodes["props"][pn]["values"])
            except Exception: pass
    edges=np.asarray(g["edges"]["ids"])
    d={"id":ids}
    for k in ("t","z","y","x"):
        if k in props: d[k]=props[k]
    return pd.DataFrame(d), edges
print("ok")

## 3 · Detection — çekirdek merkezleri
Gauss yumuşatma → Otsu eşiği → çekirdek boyutlu yerel-maksimum → bağlı bileşen ağırlık merkezi.
(EDA'da bu hat ~213 çekirdek/kare veriyordu = gerçek yoğunluk.)

In [ ]:
def detect_frame(v):
    Z,Y,X=v.shape
    sm=ndi.gaussian_filter(v.astype(np.float32), sigma=SIGMA)
    thr=threshold_otsu(sm)
    mx=ndi.maximum_filter(sm, size=FOOT)
    peaks=(sm==mx)&(sm>thr)
    lbl,n=ndi.label(peaks)
    if n==0: return np.zeros((0,3),np.float32)
    c=np.asarray(ndi.center_of_mass(sm, lbl, np.arange(1,n+1)), dtype=np.float32)  # (N,3) voxel
    if BORDER_MARGIN>0 and len(c):
        m=BORDER_MARGIN
        keep=((c[:,0]>=m)&(c[:,0]<Z-m)&(c[:,1]>=m)&(c[:,1]<Y-m)&(c[:,2]>=m)&(c[:,2]<X-m))
        c=c[keep]                              # sahte sinir tepelerini ele
    return c

# hiz testi + yogunluk kontrolu
_arr=open_image(TEST/(test_names[0]+".zarr"))
_T=_arr.shape[0]
t0=time.time(); _c=detect_frame(np.asarray(_arr[_T//2])); t1=time.time()   # T HARDCODE ETME
print(f"test shape: {_arr.shape} | 1 kare detection: {t1-t0:.2f}s | cekirdek: {len(_c)}")
print(f"tahmini 1 dataset ({_T} kare): {(t1-t0)*_T:.0f}s")
print(f"tahmini {len(test_names)} test datasi: {(t1-t0)*_T*len(test_names)/60:.1f} dk")

## 4 · Linking — Hungarian (8 µm) + bölünme (2. tur)
**1. tur:** optimal 1-1 eşleştirme; 8 µm üstü yasak. Eşleşmeyen = beliriş/kayboluş.
**2. tur (bölünme):** t+1'de eşleşmemiş bir node, t'de **zaten çocuğu olan** bir ebeveyne
≤6 µm ise → ikinci kız (ebeveyn 2-çıkışlı olur). İki kız ≤13 µm koşulu FP'yi eler.
> Detection (pahalı) linking'den ayrıldı → aynı tespitlerle **bölünmeli/bölünmesiz** karşılaştırma.

In [ ]:
def link_pairs(A,B):
    if len(A)==0 or len(B)==0: return [], None
    D=cdist(A*S, B*S)                       # um cinsinden mesafe
    cost=np.where(D<=LINK_MAX_UM, D, 1e6)   # kapi
    r,c=linear_sum_assignment(cost)
    return [(int(i),int(j)) for i,j in zip(r,c) if D[i,j]<=LINK_MAX_UM], D

def link_and_divide(A, B, div_on):
    pairs, D = link_pairs(A, B)
    if not div_on or D is None or not pairs: return pairs, []
    child_of={i:j for i,j in pairs}
    mA=np.array(sorted(child_of.keys()))
    matchedB=set(j for _,j in pairs)
    div=[]
    for j in range(len(B)):
        if j in matchedB: continue                       # eslesmis -> normal edge
        cand=mA[D[mA,j]<=DIV_MAX_UM]                      # yakin, zaten-cocuklu ebeveynler
        best_i=-1; best_d=DIV_MAX_UM+1
        for i in cand:
            sib=float(np.linalg.norm((B[child_of[i]]-B[j])*S))   # iki kiz arasi
            if D[i,j]<best_d and sib<=DIV_SIB_UM:
                best_d=D[i,j]; best_i=int(i)
        if best_i>=0: div.append((best_i,j))
    return pairs, div

def detect_all(arr):
    return [detect_frame(np.asarray(arr[t])) for t in range(arr.shape[0])]

def build_graph(cents, div_on=DIV_ON):
    nodes=[]; edges=[]; off=[]; nid=1
    for t,c in enumerate(cents):
        off.append(nid)
        for p in c:
            # SEMA: koordinatlar TAMSAYI voxel -> yuvarla
            nodes.append((nid, t, int(round(p[0])), int(round(p[1])), int(round(p[2])))); nid+=1
    for t in range(len(cents)-1):
        pairs, div = link_and_divide(cents[t], cents[t+1], div_on)
        for i,j in pairs: edges.append((off[t]+i, off[t+1]+j))
        for i,j in div:   edges.append((off[t]+i, off[t+1]+j))   # bolunme = ebeveyn 2. kiz
    return nodes, edges

def track_dataset(arr):
    return build_graph(detect_all(arr), DIV_ON)
print("ok")

## 5 · Yerel metrik — Edge Jaccard + Division Jaccard
Node'lar 7 µm ile GT'ye eşleştirilir, sonra **kenarlar** karşılaştırılır.
**Seyrek GT mantığı:** iki ucu da GT'ye eşleşmeyen tahmin kenarı **yok sayılır** (etiketsiz hücre = hata değil).
**Division (yaklaşık):** GT'de 2-çıkışlı (bölünen) node → ona eşleşen tahmin node'umuz da 2-çıkışlı mı?
Not: `edge_TP/FP` **havuzlanıp** micro alınacak (bu fonksiyon dataset başına döner).

In [ ]:
def eval_vs_gt(nodes, edges, gt_ndf, gt_edges):
    pn=pd.DataFrame(nodes, columns=["node_id","t","z","y","x"])
    gmap={}                                    # pred_id -> gt_id (7um eslesme)
    for t,g in gt_ndf.groupby("t"):
        p=pn[pn.t==int(t)]
        if len(p)==0 or len(g)==0: continue
        D=cdist(p[["z","y","x"]].values*S, g[["z","y","x"]].values*S)
        cost=np.where(D<=MATCH_UM, D, 1e6)
        r,c=linear_sum_assignment(cost)
        pid=p["node_id"].values; gid=g["id"].values
        for i,j in zip(r,c):
            if D[i,j]<=MATCH_UM: gmap[int(pid[i])]=int(gid[j])
    # --- Edge Jaccard ---
    gtset=set((int(u),int(v)) for u,v in gt_edges)
    eTP=0; eFP=0; cov=set()
    for u,v in edges:
        gu=gmap.get(u); gv=gmap.get(v)
        if gu is None or gv is None: continue  # etiketsiz -> yoksay
        if (gu,gv) in gtset: eTP+=1; cov.add((gu,gv))
        else: eFP+=1
    eFN=len(gtset)-len(cov)
    # --- Division Jaccard (yaklasik) ---
    gt_out=Counter(int(u) for u,_ in gt_edges); gt_div=set(u for u,c in gt_out.items() if c>=2)
    pr_out=Counter(u for u,_ in edges);         pr_div=set(u for u,c in pr_out.items() if c>=2)
    gt2pr={}
    for pid_,gid_ in gmap.items(): gt2pr.setdefault(gid_,pid_)
    dTP=sum(1 for gd in gt_div if gt2pr.get(gd) in pr_div)
    dFP=sum(1 for pd_ in pr_div if (pd_ in gmap) and (gmap[pd_] not in gt_div))  # eslesen ama GT'de bolunme degil
    dFN=len(gt_div)-dTP
    return dict(eTP=eTP, eFP=eFP, eFN=eFN, dTP=dTP, dFP=dFP, dFN=dFN,
                jaccard=round(eTP/max(eTP+eFP+eFN,1),4),
                node_recall=round(len(set(gmap.values()))/max(len(gt_ndf),1),4),
                pred_nodes=len(nodes), gt_div=len(gt_div))
print("ok")

### 5a · Sanity check: GT → GT skoru **1.0** olmalı

In [ ]:
# SADECE DEV: gercek rerun'da gizli test isimleri train'de YOK -> atla.
if DEV:
    g_ndf,g_edges=load_geff(TRAIN/(test_names[0]+".geff"))
    gt_nodes=[(int(r.id),int(r.t),float(r.z),float(r.y),float(r.x)) for r in g_ndf.itertuples()]
    gt_edge_list=[(int(u),int(v)) for u,v in g_edges]
    chk=eval_vs_gt(gt_nodes, gt_edge_list, g_ndf, g_edges)
    print("GT->GT:", chk)
    assert chk["jaccard"]>0.999, "SANITY FAIL — metrik implementasyonu hatali!"
    print(">> Metrik implementasyonu dogrulandi.")
else:
    print("GERCEK RERUN -> sanity check atlandi (GT yok)")

## 6 · Değerlendir — sınır temizleme + bölünmenin etkisi
Detection **bir kez** (sınır temizleme dahil), linking **iki kez** (bölünmesiz vs bölünmeli).
Resmi metrik: `Final = adj_edge_jaccard + 0.1 × division_jaccard`, hepsi **micro** (havuzlanmış).

In [ ]:
def micro_scores(rows):
    eTP=sum(r["eTP"] for r in rows); eFP=sum(r["eFP"] for r in rows); eFN=sum(r["eFN"] for r in rows)
    dTP=sum(r["dTP"] for r in rows); dFP=sum(r["dFP"] for r in rows); dFN=sum(r["dFN"] for r in rows)
    eJ=eTP/max(eTP+eFP+eFN,1); dJ=dTP/max(dTP+dFP+dFN,1)
    return dict(edge_J=eJ, div_J=dJ, final=eJ+0.1*dJ,
                eTP=eTP,eFP=eFP,eFN=eFN, dTP=dTP,dFP=dFP,dFN=dFN, gt_div=sum(r["gt_div"] for r in rows))

CACHE={}    # detection'i cache'le (pahali kisim)
if DEV:
    for nm in test_names:
        gp=TRAIN/(nm+".geff")
        if not gp.exists(): continue
        t0=time.time()
        CACHE[nm]=(detect_all(open_image(TEST/(nm+".zarr"))), load_geff(gp))
        print(f"  {nm}: detection {time.time()-t0:.0f}s")

    for div_on in [False, True]:
        rows=[]
        for nm,(cents,(gn,ge)) in CACHE.items():
            nodes,edges=build_graph(cents, div_on)
            r=eval_vs_gt(nodes,edges,gn,ge); r["dataset"]=nm; rows.append(r)
        m=micro_scores(rows)
        tag="BOLUNMELI" if div_on else "bolunmesiz"
        print(f"\n=== {tag} ===")
        for r in rows:
            print(f"  {r['dataset']}: edgeJ={r['jaccard']:.3f} recall={r['node_recall']:.3f} "
                  f"eTP={r['eTP']} eFP={r['eFP']} eFN={r['eFN']} | dTP={r['dTP']} dFP={r['dFP']} gt_div={r['gt_div']}")
        print(f"  >> MICRO edge_J={m['edge_J']:.4f} | div_J={m['div_J']:.4f} | "
              f"FINAL={m['final']:.4f}   [eTP={m['eTP']} eFP={m['eFP']} eFN={m['eFN']} | "
              f"dTP={m['dTP']} dFP={m['dFP']} dFN={m['dFN']}]")
    print("\n(referans: onceki submit edge_J=0.781, LB=0.749; bolunme yoktu)")
else:
    print("GERCEK RERUN -> degerlendirme atlandi")

## 7 · Gönderim — `test/` dinamik gez, `submission.csv` yaz
Şema: `id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`
- **node** satırı: koordinatlı, `source_id=target_id=-1`
- **edge** satırı: `node_id=t=z=y=x=-1`, `source_id→target_id`

In [ ]:
import csv
names = test_names if MAX_TEST is None else test_names[:MAX_TEST]
gid=0; tot_n=0; tot_e=0; failed=[]; t_start=time.time()
# Satir satir yaz: gizli test buyuk olabilir -> hepsini RAM'de tutma.
with open(OUT_CSV,"w",newline="") as fh:
    w=csv.writer(fh)
    w.writerow(["id","dataset","row_type","node_id","t","z","y","x","source_id","target_id"])
    for k,nm in enumerate(names,1):
        t0=time.time()
        try:
            arr=open_image(TEST/(nm+".zarr"))
            nodes,edges=track_dataset(arr)
        except Exception as e:
            # Tek dataset patlarsa TUM kosuyu oldurme; yer tutucu koy, devam et.
            print(f"  [HATA] {nm}: {type(e).__name__}: {e}")
            nodes,edges=[],[]; failed.append(nm)
        for nid,t,z,y,x in nodes:
            w.writerow([gid,nm,"node",nid,t,z,y,x,-1,-1]); gid+=1   # koordinatlar zaten int
        for u,v in edges:
            w.writerow([gid,nm,"edge",-1,-1,-1,-1,-1,u,v]); gid+=1
        if not nodes:   # SART: her test dataseti submission'da yer almali
            w.writerow([gid,nm,"node",1,0,0,0,0,-1,-1]); gid+=1
            print(f"  [uyari] {nm}: tespit yok -> yer tutucu node")
        tot_n+=len(nodes); tot_e+=len(edges)
        print(f"[{k}/{len(names)}] {nm}: node={len(nodes)} edge={len(edges)} ({time.time()-t0:.0f}s)")
print(f"\nYAZILDI: {OUT_CSV} | satir={gid:,} | node={tot_n:,} edge={tot_e:,} "
      f"| sure={(time.time()-t_start)/60:.1f} dk")
if failed: print("BASARISIZ (yer tutucu kondu):", failed)

### 7a · Şema doğrulama

In [ ]:
# Hafif kontrol (her modda): kolon semasi + ilk satirlar
head=pd.read_csv(OUT_CSV, nrows=5)
ss=ROOT/"sample_submission.csv"
if ss.exists():
    s=pd.read_csv(ss)
    print("beklenen kolonlar:", list(s.columns))
    print("bizim kolonlar   :", list(head.columns))
    assert list(head.columns)==list(s.columns), "KOLON UYUSMAZLIGI!"
print("\nilk satirlar:"); print(head.to_string(index=False))

# Agir kontrol SADECE DEV'de (gercek rerun'da dosya cok buyuk olabilir)
if DEV:
    sub=pd.read_csv(OUT_CSV)
    print("\nrow_type:", dict(sub.row_type.value_counts()))
    print("dataset :", sub.dataset.nunique(), "adet")
    missing=set(names)-set(sub.dataset.unique())
    print("eksik dataset:", missing if missing else "yok")
    assert not missing, "Bazi test datasetleri submission'da YOK!"
    nd=sub[sub.row_type=="node"]
    assert all(nd[c].map(lambda v: float(v).is_integer()).all() for c in ["t","z","y","x"]), \
        "Koordinatlar tamsayi degil!"
    print("koordinatlar tamsayi: OK")
    bad=0
    for ds,g in sub.groupby("dataset"):
        nid=set(g[g.row_type=="node"].node_id); e=g[g.row_type=="edge"]
        bad+=(~e.source_id.isin(nid)).sum()+(~e.target_id.isin(nid)).sum()
    print("gecersiz edge referansi:", bad)
    assert bad==0, "Edge referans hatasi!"
    print(">> Submission gecerli.")
else:
    print("\nGERCEK RERUN -> agir dogrulama atlandi (DEV'de yapildi)")
    print("satir sayisi:", sum(1 for _ in open(OUT_CSV))-1)

## 8 · Sonraki adımlar

Bölüm 6'daki karşılaştırmaya göre karar:
- **FINAL yükseldiyse** → submit (5/gün hakkımız var; yerel metrik LB ile tutarlı çıktı)
- **edge_J düştüyse** (bölünme FP üretmiş) → `DIV_MAX_UM`/`DIV_SIB_UM` sıkılaştır veya `DIV_ON=False`

Sıradaki büyük levaralar:
- [ ] **6bba FN azaltma** — kaybın %91'i orada; instance ayrımı (watershed / multi-threshold hipotezleri)
- [ ] **Linking kalitesi** — yoğunlukla bozuluyor (v2 tuning'de görüldü)
- [ ] **Ultrack** — asıl instance-ayrımı çözümü (yerel kontrast ~1.5×, çekirdekler temas halinde)
- [ ] **Prefix-bazlı CV** (44b6/6bba) — daha çok train örneğiyle ölç, LB'ye değil kendi skoruna güven